# NB04 — Supply, Demand, and the Market Mechanism
**Global Oil Market Analysis**

The basic economic argument: when supply exceeds demand, prices fall. When demand outstrips supply, prices rise. Oil is the most traded commodity in the world and that relationship should be visible. But the market is complicated by OPEC quotas, strategic reserves, geopolitical risk premiums, and dollar movements. This notebook tests whether the supply-demand balance actually predicts price direction.

---

## Sections
1. Global supply-demand balance over time
2. Does surplus predict falling prices? (balance vs next-month price change)
3. Which region drives demand growth?
4. China and India — the demand story of the 21st century
5. EIA forecast: supply and demand through 2027

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)

DATA_DIR = './data/processed/'

master    = pd.read_parquet(DATA_DIR + 'master_actuals.parquet')
forecast  = pd.read_parquet(DATA_DIR + 'master_forecast.parquet')
cons_reg  = pd.read_parquet(DATA_DIR + 'consumption_region.parquet')
cons_ctry = pd.read_parquet(DATA_DIR + 'consumption_country.parquet')
prices    = pd.read_parquet(DATA_DIR + 'prices.parquet')

print('Data loaded.')

Data loaded.


## 1. Global Supply-Demand Balance Over Time

In [2]:
balance = master.dropna(subset=['supply_demand_balance', 'price_world']).copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['Supply-Demand Balance (Mb/d — positive = surplus)',
                                    'World Oil Price (USD/bbl)'])

# Balance — color by surplus/deficit
colors = ['#059669' if v > 0 else '#DC2626' for v in balance['supply_demand_balance']]
fig.add_trace(go.Bar(
    x=balance.index, y=balance['supply_demand_balance'],
    marker_color=colors, name='Balance',
    showlegend=False
), row=1, col=1)
fig.add_hline(y=0, line_color='black', line_width=1, row=1, col=1)

# Price
fig.add_trace(go.Scatter(
    x=balance.index, y=balance['price_world'],
    mode='lines', line=dict(color='#1D4ED8', width=2),
    name='Price', showlegend=False
), row=2, col=1)

fig.update_layout(
    title='Global Oil Supply-Demand Balance vs Price',
    template='plotly_white', height=560, hovermode='x unified'
)
fig.add_annotation(
    x=0.01, y=0.97, xref='paper', yref='paper',
    text='Green = surplus | Red = deficit',
    showarrow=False, font=dict(size=11), bgcolor='white'
)
fig.show()

## 2. Does Surplus Predict Falling Prices?

In [3]:
# Test: does this month's balance predict next month's price change?
balance['next_month_price_chg'] = balance['price_mom_pct'].shift(-1)
clean = balance.dropna(subset=['supply_demand_balance', 'next_month_price_chg'])

corr, pvalue = stats.pearsonr(
    clean['supply_demand_balance'],
    clean['next_month_price_chg']
)
print(f'Correlation: supply-demand balance vs next-month price change')
print(f'  Pearson r = {corr:.3f}')
print(f'  p-value   = {pvalue:.4f}')
print(f'  Significant at p<0.05: {pvalue < 0.05}')
print()
print('Interpretation:')
if corr < -0.1 and pvalue < 0.05:
    print('  Surplus predicts falling prices — the market mechanism works.')
elif abs(corr) < 0.1 or pvalue >= 0.05:
    print('  Weak / insignificant relationship — other factors dominate price direction.')
else:
    print('  Positive correlation — surplus months coincide with rising prices (possibly demand-led).')

Correlation: supply-demand balance vs next-month price change
  Pearson r = -0.142
  p-value   = 0.0047
  Significant at p<0.05: True

Interpretation:
  Surplus predicts falling prices — the market mechanism works.


In [4]:
# Scatter: balance vs next-month price change
fig = px.scatter(
    clean, x='supply_demand_balance', y='next_month_price_chg',
    color='balance_direction',
    color_discrete_map={'surplus': '#059669', 'deficit': '#DC2626'},
    trendline='ols',
    labels={
        'supply_demand_balance': 'Supply-Demand Balance this month (Mb/d)',
        'next_month_price_chg': 'Price change next month (%)',
        'balance_direction': 'Balance'
    },
    title=f'Supply-Demand Balance vs Next Month Price Change (r = {corr:.3f}, p = {pvalue:.4f})'
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(template='plotly_white', height=480)
fig.show()

In [5]:
# Lag analysis — does the balance need more months to show in prices?
print('LAG ANALYSIS — correlation at different lags:')
print('(How many months until supply surplus shows up in prices?)')
print()

lag_results = []
for lag in range(0, 13):
    shifted_price = balance['price_mom_pct'].shift(-lag)
    pair = pd.DataFrame({
        'balance': balance['supply_demand_balance'],
        'price_chg': shifted_price
    }).dropna()
    r, p = stats.pearsonr(pair['balance'], pair['price_chg'])
    lag_results.append({'lag_months': lag, 'correlation': round(r, 3), 'p_value': round(p, 4)})
    sig = '*' if p < 0.05 else ''
    print(f'  Lag {lag:2d} months: r = {r:+.3f} (p = {p:.4f}) {sig}')

best = min(lag_results, key=lambda x: x['p_value'])
print()
print(f'Strongest relationship at lag {best["lag_months"]} months (r = {best["correlation"]})')

LAG ANALYSIS — correlation at different lags:
(How many months until supply surplus shows up in prices?)

  Lag  0 months: r = -0.260 (p = 0.0000) *
  Lag  1 months: r = -0.142 (p = 0.0047) *
  Lag  2 months: r = -0.046 (p = 0.3652) 
  Lag  3 months: r = -0.030 (p = 0.5466) 
  Lag  4 months: r = -0.013 (p = 0.7992) 
  Lag  5 months: r = -0.086 (p = 0.0878) 
  Lag  6 months: r = -0.066 (p = 0.1909) 
  Lag  7 months: r = -0.039 (p = 0.4385) 
  Lag  8 months: r = -0.009 (p = 0.8662) 
  Lag  9 months: r = +0.057 (p = 0.2624) 
  Lag 10 months: r = +0.067 (p = 0.1853) 
  Lag 11 months: r = +0.086 (p = 0.0920) 
  Lag 12 months: r = +0.097 (p = 0.0569) 

Strongest relationship at lag 0 months (r = -0.26)


## 3. Which Region Drives Demand Growth?

In [6]:
# Regional consumption — normalized to index (2000 = 100)
base_year = '2000'
regions = ['cons_africa', 'cons_asia_oceania', 'cons_europe',
           'cons_middle_east', 'cons_north_america', 'cons_central_south_america']

region_clean = cons_reg[regions].dropna()

try:
    base = region_clean.loc[base_year].mean()
    region_idx = region_clean.div(base) * 100
except Exception:
    # If 2000 not available, use earliest
    base = region_clean.iloc[:12].mean()
    region_idx = region_clean.div(base) * 100

labels = {
    'cons_africa':               'Africa',
    'cons_asia_oceania':         'Asia & Oceania',
    'cons_europe':               'Europe',
    'cons_middle_east':          'Middle East',
    'cons_north_america':        'North America',
    'cons_central_south_america':'C&S America',
}
colors_r = ['#DC2626','#1D4ED8','#059669','#D97706','#7C3AED','#0891B2']

fig = go.Figure()
for col, color in zip(regions, colors_r):
    fig.add_trace(go.Scatter(
        x=region_idx.index, y=region_idx[col],
        mode='lines', name=labels[col],
        line=dict(color=color, width=2)
    ))

fig.add_hline(y=100, line_dash='dash', line_color='gray', opacity=0.4)
fig.update_layout(
    title='Regional Oil Consumption Index (start = 100)',
    xaxis_title='Date', yaxis_title='Index',
    template='plotly_white', height=480, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [7]:
# Absolute growth by region
first = cons_reg[regions].dropna().iloc[0]
last  = cons_reg[regions].dropna().iloc[-1]
growth = (last - first).sort_values(ascending=False)
print('DEMAND GROWTH BY REGION (first available vs latest, Mb/d):')
for col in growth.index:
    pct = (last[col] - first[col]) / first[col] * 100
    print(f'  {labels[col]:25s}: {first[col]:.2f} -> {last[col]:.2f} ({pct:+.0f}%)')

DEMAND GROWTH BY REGION (first available vs latest, Mb/d):
  Asia & Oceania           : 14.22 -> 41.88 (+195%)
  Middle East              : 3.50 -> 9.45 (+170%)
  North America            : 20.24 -> 24.98 (+23%)
  C&S America              : 3.88 -> 7.39 (+90%)
  Africa                   : 2.07 -> 5.27 (+155%)
  Europe                   : 15.17 -> 13.97 (-8%)


## 4. China and India — The 21st Century Demand Story

In [8]:
ctry_clean = cons_ctry.dropna(subset=['cons_china', 'cons_india', 'cons_usa'])

fig = go.Figure()

for col, label, color in [
    ('cons_usa',    'United States', '#1D4ED8'),
    ('cons_china',  'China',         '#DC2626'),
    ('cons_india',  'India',         '#F59E0B'),
    ('cons_russia', 'Russia',        '#7C3AED'),
    ('cons_japan',  'Japan',         '#059669'),
]:
    data = ctry_clean[col].dropna()
    fig.add_trace(go.Scatter(
        x=data.index, y=data,
        mode='lines', name=label,
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    title='Oil Consumption — Major Economies (Mb/d)',
    xaxis_title='Date', yaxis_title='Consumption (Mb/d)',
    template='plotly_white', height=470, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [9]:
# China demand growth quantified
china_1990 = ctry_clean.loc['1990', 'cons_china'].mean()
china_2010 = ctry_clean.loc['2010', 'cons_china'].mean()
china_late = ctry_clean['cons_china'].iloc[-12:].mean()

india_1990 = ctry_clean.loc['1990', 'cons_india'].mean()
india_late = ctry_clean['cons_india'].iloc[-12:].mean()

us_1990    = ctry_clean.loc['1990', 'cons_usa'].mean()
us_late    = ctry_clean['cons_usa'].iloc[-12:].mean()

print('DEMAND GROWTH — KEY ECONOMIES:')
print(f'  China  1990: {china_1990:.2f} Mb/d  |  2010: {china_2010:.2f} Mb/d  |  Latest: {china_late:.2f} Mb/d  |  Growth: +{(china_late/china_1990-1)*100:.0f}%')
print(f'  India  1990: {india_1990:.2f} Mb/d  |  Latest: {india_late:.2f} Mb/d  |  Growth: +{(india_late/india_1990-1)*100:.0f}%')
print(f'  US     1990: {us_1990:.2f} Mb/d  |  Latest: {us_late:.2f} Mb/d  |  Growth: {(us_late/us_1990-1)*100:+.0f}%')

DEMAND GROWTH — KEY ECONOMIES:
  China  1990: 2.33 Mb/d  |  2010: 9.19 Mb/d  |  Latest: 17.04 Mb/d  |  Growth: +632%
  India  1990: 1.17 Mb/d  |  Latest: 6.27 Mb/d  |  Growth: +437%
  US     1990: 16.99 Mb/d  |  Latest: 20.73 Mb/d  |  Growth: +22%


## 5. EIA Forecast — Supply and Demand Through 2027

In [10]:
# Combine actuals + forecast for forward-looking view
# Shade the forecast period differently
combined_prod = pd.concat([master['prod_world'], forecast['prod_world']]).dropna()
combined_cons = pd.concat([master['cons_world'], forecast['cons_world']]).dropna()
forecast_start = forecast.index.min()

fig = go.Figure()

# Actuals
actual_end = master.index.max()
fig.add_trace(go.Scatter(
    x=master.dropna(subset=['prod_world']).index,
    y=master.dropna(subset=['prod_world'])['prod_world'],
    mode='lines', name='Production (actual)',
    line=dict(color='#1D4ED8', width=2)
))
fig.add_trace(go.Scatter(
    x=master.dropna(subset=['cons_world']).index,
    y=master.dropna(subset=['cons_world'])['cons_world'],
    mode='lines', name='Consumption (actual)',
    line=dict(color='#DC2626', width=2)
))

# Forecasts (dashed)
fig.add_trace(go.Scatter(
    x=forecast.dropna(subset=['prod_world']).index,
    y=forecast.dropna(subset=['prod_world'])['prod_world'],
    mode='lines', name='Production (EIA forecast)',
    line=dict(color='#93C5FD', width=2, dash='dot')
))
fig.add_trace(go.Scatter(
    x=forecast.dropna(subset=['cons_world']).index,
    y=forecast.dropna(subset=['cons_world'])['cons_world'],
    mode='lines', name='Consumption (EIA forecast)',
    line=dict(color='#FCA5A5', width=2, dash='dot')
))

# Shade the forecast period
fig.add_vrect(
    x0=forecast_start, x1=forecast.index.max(),
    fillcolor='yellow', opacity=0.05,
    annotation_text='EIA Forecast', annotation_position='top left'
)

fig.update_layout(
    title='World Oil Supply and Demand — Actuals + EIA Forecast through 2027',
    xaxis_title='Date', yaxis_title='Volume (Mb/d)',
    template='plotly_white', height=500, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [12]:
# What does the EIA forecast for the balance?
forecast_balance = forecast['prod_world'] - forecast['cons_world']
print('EIA FORECAST — SUPPLY-DEMAND BALANCE:')
for date, val in forecast_balance.dropna().resample('QE').mean().items():
    direction = 'SURPLUS' if val > 0 else 'DEFICIT'
    quarter   = pd.Timestamp(date).quarter
    label     = f'{date.year}-Q{quarter}'
    print(f'  {label:12s} : {val:+.2f} Mb/d ({direction})')

print()
print('NB04 complete. Proceed to NB05 — OPEC and the Power Shift.')

EIA FORECAST — SUPPLY-DEMAND BALANCE:
  2026-Q1      : -1.88 Mb/d (DEFICIT)
  2026-Q2      : +0.71 Mb/d (SURPLUS)
  2026-Q3      : +2.22 Mb/d (SURPLUS)
  2026-Q4      : +3.30 Mb/d (SURPLUS)
  2027-Q1      : +3.66 Mb/d (SURPLUS)
  2027-Q2      : +2.64 Mb/d (SURPLUS)
  2027-Q3      : +2.44 Mb/d (SURPLUS)
  2027-Q4      : +3.23 Mb/d (SURPLUS)

NB04 complete. Proceed to NB05 — OPEC and the Power Shift.
